# Practical Application III: Comparing Classifiers

**Overview**: In this practical application, your goal is to compare the performance of the classifiers we encountered in this section, namely K Nearest Neighbor, Logistic Regression, Decision Trees, and Support Vector Machines.  We will utilize a dataset related to marketing bank products over the telephone.  



### Getting Started

Our dataset comes from the UCI Machine Learning repository [link](https://archive.ics.uci.edu/ml/datasets/bank+marketing).  The data is from a Portugese banking institution and is a collection of the results of multiple marketing campaigns.  We will make use of the article accompanying the dataset [here](CRISP-DM-BANK.pdf) for more information on the data and features.



### Problem 1: Understanding the Data

To gain a better understanding of the data, please read the information provided in the UCI link above, and examine the **Materials and Methods** section of the paper.  How many marketing campaigns does this data represent?

### Problem 2: Read in the Data

Use pandas to read in the dataset `bank-additional-full.csv` and assign to a meaningful variable name.

In [1]:
import pandas as pd

In [3]:
df = pd.read_csv('data/bank-additional/bank-additional-full.csv', sep = ';')

In [4]:
df.head()

,age,job,marital,education,default,housing,loan,contact,month,day_of_week,...,campaign,pdays,previous,poutcome,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,y
0,56,housemaid,married,basic.4y,no,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
1,57,services,married,high.school,unknown,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
2,37,services,married,high.school,no,yes,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
3,40,admin.,married,basic.6y,no,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
4,56,services,married,high.school,no,no,yes,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no


### Problem 3: Understanding the Features


Examine the data description below, and determine if any of the features are missing values or need to be coerced to a different data type.


```
Input variables:
# bank client data:
1 - age (numeric)
2 - job : type of job (categorical: 'admin.','blue-collar','entrepreneur','housemaid','management','retired','self-employed','services','student','technician','unemployed','unknown')
3 - marital : marital status (categorical: 'divorced','married','single','unknown'; note: 'divorced' means divorced or widowed)
4 - education (categorical: 'basic.4y','basic.6y','basic.9y','high.school','illiterate','professional.course','university.degree','unknown')
5 - default: has credit in default? (categorical: 'no','yes','unknown')
6 - housing: has housing loan? (categorical: 'no','yes','unknown')
7 - loan: has personal loan? (categorical: 'no','yes','unknown')
# related with the last contact of the current campaign:
8 - contact: contact communication type (categorical: 'cellular','telephone')
9 - month: last contact month of year (categorical: 'jan', 'feb', 'mar', ..., 'nov', 'dec')
10 - day_of_week: last contact day of the week (categorical: 'mon','tue','wed','thu','fri')
11 - duration: last contact duration, in seconds (numeric). Important note: this attribute highly affects the output target (e.g., if duration=0 then y='no'). Yet, the duration is not known before a call is performed. Also, after the end of the call y is obviously known. Thus, this input should only be included for benchmark purposes and should be discarded if the intention is to have a realistic predictive model.
# other attributes:
12 - campaign: number of contacts performed during this campaign and for this client (numeric, includes last contact)
13 - pdays: number of days that passed by after the client was last contacted from a previous campaign (numeric; 999 means client was not previously contacted)
14 - previous: number of contacts performed before this campaign and for this client (numeric)
15 - poutcome: outcome of the previous marketing campaign (categorical: 'failure','nonexistent','success')
# social and economic context attributes
16 - emp.var.rate: employment variation rate - quarterly indicator (numeric)
17 - cons.price.idx: consumer price index - monthly indicator (numeric)
18 - cons.conf.idx: consumer confidence index - monthly indicator (numeric)
19 - euribor3m: euribor 3 month rate - daily indicator (numeric)
20 - nr.employed: number of employees - quarterly indicator (numeric)

Output variable (desired target):
21 - y - has the client subscribed a term deposit? (binary: 'yes','no')
```



### Problem 4: Understanding the Task

After examining the description and data, your goal now is to clearly state the *Business Objective* of the task.  State the objective below.

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 41188 entries, 0 to 41187
Data columns (total 21 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   age             41188 non-null  int64  
 1   job             41188 non-null  object 
 2   marital         41188 non-null  object 
 3   education       41188 non-null  object 
 4   default         41188 non-null  object 
 5   housing         41188 non-null  object 
 6   loan            41188 non-null  object 
 7   contact         41188 non-null  object 
 8   month           41188 non-null  object 
 9   day_of_week     41188 non-null  object 
 10  duration        41188 non-null  int64  
 11  campaign        41188 non-null  int64  
 12  pdays           41188 non-null  int64  
 13  previous        41188 non-null  int64  
 14  poutcome        41188 non-null  object 
 15  emp.var.rate    41188 non-null  float64
 16  cons.price.idx  41188 non-null  float64
 17  cons.conf.idx   41188 non-null 

In [ ]:
'''
Business Objective

The business objective is to build and compare multiple classification models (KNN, Logistic Regression, Decision Tree, and SVM). 
These model can predict whether a customer will subscribe to the bank’s term deposit product (“y” = yes/no) based on information 
available from a telephone marketing campaign (customer demographics, previous campaign outcomes, contact details, and economic indicators).
'''

### Problem 5: Engineering Features

Now that you understand your business objective, we will build a basic model to get started.  Before we can do this, we must work to encode the data.  Using just the bank information features, prepare the features and target column for modeling with appropriate encoding and transformations.

In [ ]:
# 1) Features and Target

In [6]:
X = df.drop(columns=["y"])  # all bank info features
y = df["y"].map({"no": 0, "yes": 1})  # binary encode target

In [ ]:
# 2) Separate numeric vs categorical

In [7]:
numeric_features = X.select_dtypes(include=["int64", "float64"]).columns
categorical_features = X.select_dtypes(include=["object"]).columns

In [ ]:
# 3) Build preprocessing pipelines

In [9]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

In [15]:
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ]
)

### Problem 6: Train/Test Split

With your data prepared, split it into a train and test set.

In [16]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

### Problem 7: A Baseline Model

Before we build our first model, we want to establish a baseline.  What is the baseline performance that our classifier should aim to beat?

In [17]:
class_dist = df["y"].value_counts(normalize=True)
print("Class distribution:\n", class_dist)

Class distribution:
 y
no     0.887346
yes    0.112654
Name: proportion, dtype: float64


In [18]:
baseline_accuracy = class_dist.max()
print("\nBaseline accuracy (majority class):", baseline_accuracy)


Baseline accuracy (majority class): 0.8873458288821987


### Problem 8: A Simple Model

Use Logistic Regression to build a basic model on your data.  

In [20]:
from sklearn.linear_model import LogisticRegression
log_reg_model = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("clf", LogisticRegression(max_iter=1000))
])

In [21]:
log_reg_model.fit(X_train, y_train)

,steps,"[('preprocess', ...), ('clf', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


### Problem 9: Score the Model

What is the accuracy of your model?

In [23]:
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, roc_auc_score

y_pred = log_reg_model.predict(X_test)
y_proba = log_reg_model.predict_proba(X_test)[:, 1]

acc = accuracy_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_proba)
cm = confusion_matrix(y_test, y_pred)

print(f"Logistic Regression Accuracy: {acc:.4f}")
print(f"Logistic Regression ROC-AUC:  {roc_auc:.4f}")
print("\nConfusion Matrix [[TN FP], [FN TP]]:\n", cm)
print("\nClassification Report:\n", classification_report(y_test, y_pred, digits=4))

Logistic Regression Accuracy: 0.9166
Logistic Regression ROC-AUC:  0.9424

Confusion Matrix [[TN FP], [FN TP]]:
 [[7146  164]
 [ 523  405]]

Classification Report:
               precision    recall  f1-score   support

           0     0.9318    0.9776    0.9541      7310
           1     0.7118    0.4364    0.5411       928

    accuracy                         0.9166      8238
   macro avg     0.8218    0.7070    0.7476      8238
weighted avg     0.9070    0.9166    0.9076      8238



### Problem 10: Model Comparisons

Now, we aim to compare the performance of the Logistic Regression model to our KNN algorithm, Decision Tree, and SVM models.  Using the default settings for each of the models, fit and score each.  Also, be sure to compare the fit time of each of the models.  Present your findings in a `DataFrame` similar to that below:

| Model | Train Time | Train Accuracy | Test Accuracy |
| ----- | ---------- | -------------  | -----------   |
|     |    |.     |.     |

In [28]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
import time
from sklearn.metrics import accuracy_score

In [26]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "KNN": KNeighborsClassifier(),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    # Keep SVC default settings but add max_iter to prevent extremely long runtime
    "SVM (SVC)": SVC(max_iter=2000)
}

In [30]:
results = []
for name, model in models.items():
    clf = Pipeline(steps=[("preprocess", preprocessor),
                         ("model", model)])
    start = time.time()
    clf.fit(X_train, y_train)
    train_time = time.time() - start
    
    train_acc = accuracy_score(y_train, clf.predict(X_train))
    test_acc = accuracy_score(y_test, clf.predict(X_test))
    
    results.append({
        "Model": name,
        "Train Time (s)": train_time,
        "Train Accuracy": train_acc,
        "Test Accuracy": test_acc
    })

/lib/python3.13/site-packages/threadpoolctl.py:1123: RuntimeWarning: JsProxy.as_object_map() is deprecated. Use as_py_json() instead.
  for filepath in LDSO.loadedLibsByName.as_object_map():
/lib/python3.13/site-packages/sklearn/svm/_base.py:305: ConvergenceWarning: Solver terminated early (max_iter=2000).  Consider pre-processing your data with StandardScaler or MinMaxScaler.
  warnings.warn(


In [31]:
results_df = pd.DataFrame(results)
results_df["Train Time (s)"] = results_df["Train Time (s)"].round(4)
results_df["Train Accuracy"] = results_df["Train Accuracy"].round(4)
results_df["Test Accuracy"] = results_df["Test Accuracy"].round(4)

In [32]:
results_df.sort_values("Test Accuracy", ascending=False).reset_index(drop=True)

,Model,Train Time (s),Train Accuracy,Test Accuracy
0,Logistic Regression,1.417,0.9102,0.9166
1,KNN,0.253,0.9277,0.9076
2,Decision Tree,0.655,1.0000,0.8946
3,SVM (SVC),68.772,0.8782,0.8696


### Problem 11: Improving the Model

Now that we have some basic models on the board, we want to try to improve these.  Below, we list a few things to explore in this pursuit.


- Hyperparameter tuning and grid search.  All of our models have additional hyperparameters to tune and explore.  For example the number of neighbors in KNN or the maximum depth of a Decision Tree.  
- Adjust your performance metric

In [36]:
def run_gridsearch_compare_models(
    X_train, y_train, X_test, y_test,
    scoring="f1",              # good default for imbalanced target
    cv=5,
    n_jobs=-1,
    random_state=42,
    verbose=0,
    tuning_sample_size=None,   # set e.g. 5000 to speed up KNN/SVC tuning
    param_grids=None
):
    """
    Runs GridSearchCV for 4 models (KNN, Logistic Regression, Decision Tree, SVM)
    and returns a comparison table with:
      Best Params | CV Score | Test Accuracy | Test F1 | Recall | Train Time

    Notes:
    - If runtime is slow, set tuning_sample_size (e.g., 5000) and/or reduce cv/grids.
    - SVM uses sklearn.svm.SVC (default settings) unless you change its grid.
    """

    # ---------- Preprocess once ----------
    num_cols = X_train.select_dtypes(include=["int64", "float64"]).columns
    cat_cols = X_train.select_dtypes(include=["object"]).columns

    preprocessor = ColumnTransformer(
        transformers=[
            ("num", Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler())
            ]), num_cols),
            ("cat", Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("onehot", OneHotEncoder(handle_unknown="ignore"))
            ]), cat_cols),
        ]
    )

    X_train_p = preprocessor.fit_transform(X_train)
    X_test_p  = preprocessor.transform(X_test)

    # ---------- Optional stratified subsample for tuning ----------
    if tuning_sample_size is not None and tuning_sample_size < X_train_p.shape[0]:
        rng = np.random.RandomState(random_state)
        y_arr = np.asarray(y_train)

        idx0 = np.where(y_arr == 0)[0]
        idx1 = np.where(y_arr == 1)[0]

        # keep original class ratio
        n1 = max(1, int(tuning_sample_size * (len(idx1) / len(y_arr))))
        n0 = tuning_sample_size - n1

        sub0 = rng.choice(idx0, size=min(n0, len(idx0)), replace=False)
        sub1 = rng.choice(idx1, size=min(n1, len(idx1)), replace=False)

        sub_idx = np.concatenate([sub0, sub1])
        rng.shuffle(sub_idx)

        X_tune = X_train_p[sub_idx]
        y_tune = y_train.iloc[sub_idx] if hasattr(y_train, "iloc") else y_arr[sub_idx]
    else:
        X_tune, y_tune = X_train_p, y_train

    # ---------- Default grids (edit freely) ----------
    if param_grids is None:
        param_grids = {
            "KNN": {
                "n_neighbors": [5, 11, 21],
                "weights": ["uniform", "distance"],
                "p": [1, 2],
            },
            "Logistic Regression": {
                "C": [0.1, 1, 10],
                "penalty": ["l2"],
                "solver": ["lbfgs"],
            },
            "Decision Tree": {
                "max_depth": [3, 5, 10, None],
                "min_samples_leaf": [1, 5, 10],
                "min_samples_split": [2, 10, 20],
            },
            "SVM": {
                "C": [0.1, 1, 10],
                "kernel": ["rbf"],         # keep default kernel; add "linear" if you want
                "gamma": ["scale"],        # default gamma behavior
            }
        }

    models = {
        "KNN": KNeighborsClassifier(),
        "Logistic Regression": LogisticRegression(max_iter=2000),
        "Decision Tree": DecisionTreeClassifier(random_state=random_state),
        "SVM": SVC()  # default SVC settings
    }

    skf = StratifiedKFold(n_splits=cv, shuffle=True, random_state=random_state)

    rows = []
    for name, model in models.items():
        grid = GridSearchCV(
            estimator=model,
            param_grid=param_grids[name],
            scoring=scoring,
            cv=skf,
            n_jobs=n_jobs,
            verbose=verbose,
            refit=True
        )

        t0 = time.perf_counter()
        grid.fit(X_tune, y_tune)
        train_time = time.perf_counter() - t0

        # Refit best model on FULL training data (fair test evaluation)
        best_model = grid.best_estimator_
        best_model.fit(X_train_p, y_train)

        y_pred_test = best_model.predict(X_test_p)

        rows.append({
            "Model": name,
            "Train Time": train_time,
            "Best Params": grid.best_params_,
            "CV Score": grid.best_score_,
            "Test Accuracy": accuracy_score(y_test, y_pred_test),
            "Test F1": f1_score(y_test, y_pred_test, pos_label=1),
            "Recall": recall_score(y_test, y_pred_test, pos_label=1),
        })

    out = pd.DataFrame(rows)
    out["Train Time"] = out["Train Time"].round(2)
    out["CV Score"] = out["CV Score"].round(4)
    out["Test Accuracy"] = out["Test Accuracy"].round(4)
    out["Test F1"] = out["Test F1"].round(4)
    out["Recall"] = out["Recall"].round(4)

    # order like you asked
    out = out[["Model", "Train Time", "Best Params", "CV Score", "Test Accuracy", "Test F1", "Recall"]]
    return out.sort_values("CV Score", ascending=False).reset_index(drop=True)


# ----------------------------
# Example usage
# ----------------------------
# Tip: If GridSearch takes too long, set tuning_sample_size=5000 and/or cv=3.
results_df = run_gridsearch_compare_models(
    X_train, y_train, X_test, y_test,
    scoring="f1",
    cv=3,
    n_jobs=-1,
    tuning_sample_size=5000
)

results_df


,Model,Train Time,Best Params,CV Score,Test Accuracy,Test F1,Recall
0,Decision Tree,2.28,"{'max_depth': 3, 'min_samples_leaf': 5, 'min_s...",0.5471,0.9133,0.6060,0.5916
1,Logistic Regression,0.80,"{'C': 10, 'penalty': 'l2', 'solver': 'lbfgs'}",0.4654,0.9164,0.5404,0.4364
2,SVM,41.87,"{'C': 10, 'gamma': 'scale', 'kernel': 'rbf'}",0.4589,0.9104,0.5416,0.4698
3,KNN,56.83,"{'n_neighbors': 5, 'p': 2, 'weights': 'uniform'}",0.4083,0.9076,0.5217,0.4472


##### Questions